### Необходимые импорты

In [1]:
# imports

import sys
import os
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, OrderedDict
from datetime import datetime
from collections import OrderedDict as ODict

from pydrake.all import (
    AddDefaultVisualization,
    AddMultibodyPlantSceneGraph,
    CompositeTrajectory,
    Context,
    Diagram,
    DiagramBuilder,
    GraphOfConvexSetsOptions,
    HPolyhedron,
    IrisOptions,
    MultibodyPlant,
    Parser,
    IrisNp,
    Rgba,
    RigidTransform,
    Solve,
    Sphere,
    StartMeshcat,
    RollPitchYaw,
    SpatialInertia,
    UnitInertia,
    CoulombFriction,
    Box,
    Rgba,
    FindResourceOrThrow,
)
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.geometry.optimization import Point

from manipulation.scenarios import AddIiwa, AddWsg
from manipulation.utils import ConfigureParser
from utils import solve_IK

### Настраиваем сцены

In [45]:
# сцена 1
builder = DiagramBuilder()
        
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)
iiwa = AddIiwa(plant)
wsg = AddWsg(plant, iiwa, welded=True, sphere=False)

parser = Parser(plant)
ConfigureParser(parser)
bin_model = parser.AddModelsFromUrl("package://manipulation/shelves.sdf")[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("shelves_body", bin_model),
    RigidTransform([0.88, 0, 0.4]),
)

plant.Finalize()

meshcat1 = StartMeshcat()

AddDefaultVisualization(builder, meshcat1)
diagram = builder.Build()
context = diagram.CreateDefaultContext()
plant = diagram.GetSubsystemByName("plant")
plant_context = plant.GetMyContextFromRoot(context)
diagram.ForcedPublish(context)

INFO:drake:Meshcat listening for connections at http://localhost:7003


In [2]:
from pydrake.all import LoadIrisRegionsYamlFile
single_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_10_1.yaml")
single_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_20_1.yaml")
single_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_30_1.yaml")
single_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_50_1.yaml")

single = [single_10, single_20, single_30, single_50]

two_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_10_1.yaml")
two_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_20_1.yaml")
two_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_30_1.yaml")
two_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_50_1.yaml")

two = [two_10, two_20, two_30, two_50]

three_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TABLE_THREE_SHELVES_10_1.yaml")
three_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TABLE_THREE_SHELVES_20_1.yaml")
three_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TABLE_THREE_SHELVES_30_1.yaml")
three_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TABLE_THREE_SHELVES_50_1.yaml")

three = [three_10, three_20, three_30, three_50]


In [3]:
def is_collision_free(diagram, context, plant, q) -> bool:
    plant_context = plant.GetMyContextFromRoot(context)
    plant.SetPositions(plant_context, q)
    sg = diagram.GetSubsystemByName("scene_graph")
    sg_context = sg.GetMyContextFromRoot(context)
    query = sg.get_query_output_port().Eval(sg_context)
    return not query.HasCollisions()

In [4]:
def sample_collision_free(diagram, context, plant, n, rng) -> List[np.ndarray]:
    q_lo = plant.GetPositionLowerLimits()
    q_hi = plant.GetPositionUpperLimits()
    configs = []
    for _ in range(n * 50):
        q = rng.uniform(q_lo, q_hi)
        if is_collision_free(diagram, context, plant, q):
            configs.append(q.copy())
            if len(configs) >= n:
                break
    return configs

In [5]:
def estimate_coverage(regions: List[HPolyhedron], test_points: List[np.ndarray]) -> float:
    """Estimate coverage: fraction of test points inside at least one region."""
    if not regions or not test_points:
        return 0.0
    
    covered = 0
    for q in test_points:
        for region in regions:
            if region.PointInSet(q):
                covered += 1
                break
    return covered / len(test_points)

In [13]:
from utils import solve_IK
import random 

def check_point_in_regions(point, regions):
    for key, value in regions.items():
        if not value.PointInSet(np.array(point)):
            return False
    return True

regions = [value for key, value in three_50.items()]
rng_coverage = np.random.default_rng(42 + 1000 + 50)
collision_free = sample_collision_free(diagram, context, plant, 50, rng_coverage)
print("COVERAGE   ", estimate_coverage(regions, collision_free))

for i, point in enumerate(collision_free):
    plant.SetPositions(plant_context, point)

    gripper_frame = plant.GetFrameByName("body", wsg)
    X_WG = plant.CalcRelativeTransform(
        plant_context,
        plant.world_frame(),
        gripper_frame
    )
    position = X_WG.translation()
    rotation = X_WG.rotation()

    # start_point = [0.35, 0.1, 0.55]
    # end_point = [2, 2, 2]

    X_Wp = RigidTransform(position)
    meshcat.SetObject(
        f'point{i}',
        Sphere(0.02),
        rgba=Rgba(random.random(), random.random(), random.random(), 1),
    )
    meshcat.SetTransform(f'point{i}', X_Wp)

# X_Wp = RigidTransform(end_point)
# meshcat1.SetObject(
#     "end",
#     Sphere(0.02),
#     rgba=Rgba(random.random(), random.random(), random.random(), 1),
# )
# meshcat1.SetTransform("end", X_Wp)

# q_safe = plant.GetPositions(plant_context, iiwa)
# gripper_frame = plant.GetFrameByName("body", wsg)

# start_point_IK = solve_IK(plant, plant_context, start_point, gripper_frame, q_safe)
# end_point_IK = solve_IK(plant, plant_context, end_point, gripper_frame, q_safe)

# print(is_collision_free(diagram, context, plant, q_safe))
# print(is_collision_free(diagram, context, plant, q_safe))

# print(check_point_in_regions(collision_free[0], single_50))
# print(check_point_in_regions(end_point_IK, single_50))

# for i in collision_free:
#     print(check_point_in_regions(i, single_50))


COVERAGE    0.7


In [16]:
# сцена 2

builder = DiagramBuilder()

plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)

iiwa = AddIiwa(plant)
wsg = AddWsg(plant, iiwa, welded=True, sphere=False)

parser = Parser(plant)
ConfigureParser(parser)

# Полка 1
shelves1 = parser.AddModelsFromUrl("package://manipulation/shelves.sdf")[0]
plant.RenameModelInstance(shelves1, "shelves1")

# Полка 2
shelves2 = parser.AddModelsFromUrl("package://manipulation/shelves.sdf")[0]
plant.RenameModelInstance(shelves2, "shelves2")

x  = 0.95
y1 = -0.35
y2 = +0.35
z  = 0.40
yaw = 0.0

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("shelves_body", shelves1),
    RigidTransform(RollPitchYaw(0.0, 0.0, yaw), [x, y1, z]),
)

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("shelves_body", shelves2),
    RigidTransform(RollPitchYaw(0.0, 0.0, yaw), [x, y2, z]),
)

plant.Finalize()

meshcat2 = StartMeshcat()
AddDefaultVisualization(builder, meshcat2)

diagram = builder.Build()
context = diagram.CreateDefaultContext()

plant = diagram.GetSubsystemByName("plant")
plant_context = plant.GetMyContextFromRoot(context)

q_safe = np.array([0.0, 0.6, 0.0, -1.2, 0.0, 1.0, 0.0])
plant.SetPositions(plant_context, iiwa, q_safe)

diagram.ForcedPublish(context)



INFO:drake:Meshcat listening for connections at http://localhost:7001


In [6]:
import numpy as np

from pydrake.all import (
    DiagramBuilder, RigidTransform, RollPitchYaw,
    SpatialInertia, UnitInertia, CoulombFriction
)
from pydrake.geometry import Box
from pydrake.multibody.parsing import Parser
from manipulation.scenarios import AddMultibodyPlantSceneGraph, ConfigureParser


def WeldByBodyName(plant, model_instance, body_name, X_WB):
    body = plant.GetBodyByName(body_name, model_instance)
    plant.WeldFrames(plant.world_frame(), body.body_frame(), X_WB)


def AddTableWithLegs(plant, name, size_xy, top_thickness, top_z, leg_xy, rgba4):
    lx, ly = size_xy
    legx, legy = leg_xy
    leg_h = max(0.05, top_z - top_thickness)

    mass = 30.0
    inertia = SpatialInertia(
        mass=mass,
        p_PScm_E=[0.0, 0.0, 0.0],
        G_SP_E=UnitInertia.SolidBox(lx, ly, top_thickness),
    )
    body = plant.AddRigidBody(name, inertia)

    X_WB = RigidTransform([0.0, 0.0, top_z - 0.5 * top_thickness])
    plant.WeldFrames(plant.world_frame(), body.body_frame(), X_WB)

    friction = CoulombFriction(0.9, 0.8)

    top_shape = Box(lx, ly, top_thickness)
    plant.RegisterCollisionGeometry(body, RigidTransform(), top_shape, f"{name}_top_collision", friction)
    plant.RegisterVisualGeometry(body, RigidTransform(), top_shape, f"{name}_top_visual", rgba4)

    leg_shape = Box(legx, legy, leg_h)
    x_off = 0.5 * lx - 0.5 * legx
    y_off = 0.5 * ly - 0.5 * legy
    z_off = -(0.5 * top_thickness + 0.5 * leg_h)

    for i, (sx, sy) in enumerate([(1, 1), (1, -1), (-1, 1), (-1, -1)]):
        X_BL = RigidTransform([sx * x_off, sy * y_off, z_off])
        plant.RegisterCollisionGeometry(body, X_BL, leg_shape, f"{name}_leg{i}_collision", friction)
        plant.RegisterVisualGeometry(body, X_BL, leg_shape, f"{name}_leg{i}_visual", rgba4)


def AddShelvesWelded(plant, parser, name, xyz, yaw_deg):
    m = parser.AddModelsFromUrl("package://manipulation/shelves.sdf")[0]
    plant.RenameModelInstance(m, name)
    X_WB = RigidTransform(RollPitchYaw(0.0, 0.0, np.deg2rad(yaw_deg)), xyz)
    WeldByBodyName(plant, m, "shelves_body", X_WB)
    return m


builder = DiagramBuilder()
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)

z_table_top = 0.75
table_color = np.array([0.55, 0.25, 0.10, 1.0])

AddTableWithLegs(
    plant,
    name="table",
    size_xy=(2.30, 1.80),
    top_thickness=0.10,
    top_z=z_table_top,
    leg_xy=(0.08, 0.08),
    rgba4=table_color,
)

parser = Parser(plant)
ConfigureParser(parser)

iiwa = parser.AddModelsFromUrl("package://drake_models/iiwa_description/sdf/iiwa14_no_collision.sdf")[0]
plant.RenameModelInstance(iiwa, "iiwa14")
WeldByBodyName(plant, iiwa, "iiwa_link_0", RigidTransform([0.0, 0.0, z_table_top]))

wsg = AddWsg(plant, iiwa, welded=True, sphere=False)

z_shelves = z_table_top + 0.40

x_side = 0.85
y_side = 0.65
AddShelvesWelded(plant, parser, "shelves_left_45",  [x_side, +y_side, z_shelves], +45.0)
AddShelvesWelded(plant, parser, "shelves_right_45", [x_side, -y_side, z_shelves], -45.0)

x_back = -0.95
AddShelvesWelded(plant, parser, "shelves_back", [x_back, 0.0, z_shelves], 180.0)

plant.Finalize()

meshcat = StartMeshcat()
AddDefaultVisualization(builder, meshcat)

diagram = builder.Build()
context = diagram.CreateDefaultContext()
plant = diagram.GetSubsystemByName("plant")
plant_context = plant.GetMyContextFromRoot(context)

q_safe = np.array([0.0, 0.6, 0.0, -1.2, 0.0, 1.0, 0.0])
plant.SetPositions(plant_context, iiwa, q_safe)

diagram.ForcedPublish(context)


INFO:drake:Meshcat listening for connections at http://localhost:7001


In [ ]:
from pydrake.all import LoadIrisRegionsYamlFile
single_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_10_1.yaml")
single_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_20_1.yaml")
single_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_30_1.yaml")
single_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/SINGLE_SHELF_50_1.yaml")

single = [single_10, single_20, single_30, single_50]

two_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_10_1.yaml")
two_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_20_1.yaml")
two_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_30_1.yaml")
two_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/TWO_SHELVES_50_1.yaml")

two = [two_10, two_20, two_30, two_50]

three_10 = LoadIrisRegionsYamlFile("../experiments/iris_regions/THREE_SHELVES_10_1.yaml")
three_20 = LoadIrisRegionsYamlFile("../experiments/iris_regions/THREE_SHELVES_20_1.yaml")
three_30 = LoadIrisRegionsYamlFile("../experiments/iris_regions/THREE_SHELVES_30_1.yaml")
three_50 = LoadIrisRegionsYamlFile("../experiments/iris_regions/THREE_SHELVES_50_1.yaml")

three = [three_10, three_20, three_30, three_50]
